# XỬ LÝ LỖI DỮ LIỆU — TECHX ACADEMY

File này tiếp nối file **kiểm tra chất lượng dữ liệu**. Với mỗi lỗi đã phát hiện, notebook trả lời 3 câu hỏi trước khi xử lý:
1. **Cột nào sai, nguyên nhân thật sự là gì** (lỗi dấu/chính tả hay lỗi bản chất khác)?
2. **Có thể thay thế bằng giá trị gì** một cách có căn cứ, không suy đoán bừa?
3. **Ngày âm** là do lỗi logic khách hàng hay do bối cảnh nghiệp vụ/hệ thống?

Kết luận của mỗi bước được trình bày ở ô **Markdown ngay bên dưới** ô code tương ứng (thay vì viết dưới dạng comment `#` trong code).

Nguồn dữ liệu: `Courses of Interest Table.xlsx` (Sheet1).

In [1]:
import pandas as pd
import numpy as np

INPUT_FILE = "Courses of Interest Table.xlsx"
OUTPUT_XLSX = "Courses_of_Interest_Processed.xlsx"
OUTPUT_CSV = "Courses_of_Interest_Processed.csv"
SHEET_NAME = "Sheet1"

def print_header(title):
    if ":" in title:
        title = title.split(":", 1)[1].strip()
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

In [2]:
print_header("CELL 1: ĐỌC DỮ LIỆU")

df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)
df.columns = df.columns.str.strip()

df["STT"] = pd.to_numeric(df["STT"], errors="coerce")
df = df[df["STT"].notna()].copy()
df["STT"] = df["STT"].astype(int)

zero_cols = [
    "quan_tam_khoa_hoc",
    "thoi_gian_chuyen_sang_tu_van",
    "thoi_gian_chuyen_sang_chot_sale",
    "hieu_suat",
    "quan_tam_khoa_hoc_1",
]
for col in zero_cols:
    df[col] = df[col].replace([0, "0", ""], np.nan)


ĐỌC DỮ LIỆU


In [3]:
for col in ["ngay_tao", "thoi_gian_chuyen_sang_tu_van", "thoi_gian_chuyen_sang_chot_sale"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["hieu_suat"] = pd.to_numeric(df["hieu_suat"], errors="coerce")
df["hieu_suat_date"] = pd.to_datetime(
    df["hieu_suat"].where(df["hieu_suat"] > 40000),
    unit="D",
    origin="1899-12-30",
)
df["hieu_suat_days"] = df["hieu_suat"].where(
    (df["hieu_suat"] > 0) & (df["hieu_suat"] <= 40000)
)

df["so_ngay_tu_van_den_chot"] = (
    df["thoi_gian_chuyen_sang_chot_sale"]
    - df["thoi_gian_chuyen_sang_tu_van"]
).dt.days

In [4]:
print_header("CELL 1: CHUẨN HÓA GIÁ TRỊ")

df["sentimatic"] = df["sentimatic"].replace({"unknow": "unknown"})
df["nha_mang"] = df["nha_mang"].replace(
    {"Mobi": "Mobifone", "Vina": "Vinaphone"}
)
df["MKT_xu_ly_data"] = df["MKT_xu_ly_data"].replace([0, "0"], np.nan)
df["quan_tam_khoa_hoc"] = df["quan_tam_khoa_hoc"].replace(
    {"TechX Cetificate": "TechX Certificate"}
)


CHUẨN HÓA GIÁ TRỊ


In [5]:
print_header("CELL 1: TẠO BẢNG MAPPING")

mapping_chuan = {
    "Full time": "Khóa A",
    "Part time": "Khóa B",
    "Fukuoka": "Khóa C",
    "IT Tsunami": "Khóa C",
    "TechX Certificate": "Khóa D",
    "PCIC": "Khóa E",
    "Data Analyst": "Khóa F",
}
df["khoa_hoc_du_kien"] = df["quan_tam_khoa_hoc"].map(mapping_chuan)


TẠO BẢNG MAPPING


In [6]:
print_header("CELL 1: TẠO CỜ LỖI MAPPING")

df["loi_mapping_khoa"] = (
    df["khoa_hoc_du_kien"].notna()
    & df["quan_tam_khoa_hoc_1"].notna()
    & (df["khoa_hoc_du_kien"] != df["quan_tam_khoa_hoc_1"])
)
df["khoa_hoc_la"] = (
    df["quan_tam_khoa_hoc"].notna()
    & ~df["quan_tam_khoa_hoc"].isin(mapping_chuan)
)


TẠO CỜ LỖI MAPPING


In [7]:
print_header("CELL 1: KIỂM TRA NGÀY")

df["loi_ngay_am"] = df["so_ngay_tu_van_den_chot"] < 0
df["loi_chot_truoc_tao"] = (
    df["thoi_gian_chuyen_sang_chot_sale"] < df["ngay_tao"]
)
df["loi_tu_van_truoc_tao"] = (
    df["thoi_gian_chuyen_sang_tu_van"] < df["ngay_tao"]
)
today = pd.Timestamp.today().normalize()
df["loi_ngay_tuong_lai"] = (
    (df["ngay_tao"] > today)
    | (df["thoi_gian_chuyen_sang_tu_van"] > today)
    | (df["thoi_gian_chuyen_sang_chot_sale"] > today)
)


KIỂM TRA NGÀY


In [8]:
print_header("CELL 1: KIỂM TRA NHÀ MẠNG VÀ SENTIMENT")

nha_mang_hop_le = {"Viettel", "Vinaphone", "Mobifone", "Khác"}
sentimatic_hop_le = {"tich_cuc", "tieu_cuc", "unknown"}
df["loi_nha_mang"] = (
    df["nha_mang"].notna()
    & ~df["nha_mang"].isin(nha_mang_hop_le)
)
df["loi_sentimatic"] = (
    df["sentimatic"].notna()
    & ~df["sentimatic"].isin(sentimatic_hop_le)
)


KIỂM TRA NHÀ MẠNG VÀ SENTIMENT


In [9]:
print_header("CELL 1: TÁCH TÊN TỪ ID")

df[["khach_hang", "tu_van_vien"]] = df["ID"].astype(str).str.rsplit(
    "-", n=1, expand=True
)
df["khach_hang"] = df["khach_hang"].str.strip()
df["tu_van_vien"] = df["tu_van_vien"].str.strip()
df["loi_ten_rong"] = (
    df["khach_hang"].fillna("").str.strip().eq("")
    | df["tu_van_vien"].fillna("").str.strip().eq("")
)

print(
    f"Đã xử lý {len(df)} dòng và tạo {df.shape[1]} cột."
)


TÁCH TÊN TỪ ID
Đã xử lý 18978 dòng và tạo 28 cột.


### Kết luận: Đọc dữ liệu và tạo cờ kiểm tra

Dữ liệu đã có đầy đủ các cờ kiểm tra như notebook đánh giá chất lượng.

In [10]:
print_header("CELL 2: BẢNG TỔNG HỢP LỖI THEO CỘT")

summary_rows = [
    ("nha_mang", "loi_nha_mang", df["loi_nha_mang"].sum()),
    ("sentimatic", "loi_sentimatic", df["loi_sentimatic"].sum()),
    ("quan_tam_khoa_hoc", "khoa_hoc_la", df["khoa_hoc_la"].sum()),
    ("quan_tam_khoa_hoc_1", "loi_mapping_khoa", df["loi_mapping_khoa"].sum()),
    ("so_ngay_tu_van_den_chot", "loi_ngay_am", df["loi_ngay_am"].sum()),
    ("thoi_gian_chuyen_sang_chot_sale", "loi_chot_truoc_tao", df["loi_chot_truoc_tao"].sum()),
    ("thoi_gian_chuyen_sang_tu_van", "loi_tu_van_truoc_tao", df["loi_tu_van_truoc_tao"].sum()),
    ("ngay_tao / thoi_gian_*", "loi_ngay_tuong_lai", df["loi_ngay_tuong_lai"].sum()),
    ("khach_hang / tu_van_vien", "loi_ten_rong", df["loi_ten_rong"].sum()),
]
df_summary = pd.DataFrame(summary_rows, columns=["Cột dữ liệu", "Cờ lỗi", "Số dòng lỗi"])
df_summary = df_summary.sort_values("Số dòng lỗi", ascending=False)
print(df_summary.to_string(index=False))



BẢNG TỔNG HỢP LỖI THEO CỘT
                    Cột dữ liệu               Cờ lỗi  Số dòng lỗi
       khach_hang / tu_van_vien         loi_ten_rong          709
        so_ngay_tu_van_den_chot          loi_ngay_am          164
              quan_tam_khoa_hoc          khoa_hoc_la           35
            quan_tam_khoa_hoc_1     loi_mapping_khoa            3
                       nha_mang         loi_nha_mang            2
                     sentimatic       loi_sentimatic            0
thoi_gian_chuyen_sang_chot_sale   loi_chot_truoc_tao            0
   thoi_gian_chuyen_sang_tu_van loi_tu_van_truoc_tao            0
         ngay_tao / thoi_gian_*   loi_ngay_tuong_lai            0


### Kết quả: Tổng hợp lỗi theo cột

- Thiếu tên khách hàng hoặc tư vấn viên: 709 dòng.
- Khoảng cách ngày âm: 164 dòng.
- Tên khóa chưa nhận diện ban đầu: 35 dòng; 22 dòng `AI` có mã `Khóa G`, còn 13 dòng cần xác nhận.
- Mã khóa gán lệch: 3 dòng; giá trị nhà mạng không hợp lệ: 2 dòng.
- Các nhóm lỗi ngày khác và sentiment: 0 dòng.

In [11]:
print_header("CELL 3: XỬ LÝ NHÀ MẠNG")

print("Các dòng lỗi trước khi xử lý:")
print(df.loc[df["loi_nha_mang"], ["STT", "ID", "nha_mang"]].to_string(index=False))

df["nha_mang"] = df["nha_mang"].replace(
    {0: "Không xác định", "0": "Không xác định"}
)
df["loi_nha_mang"] = (
    df["nha_mang"].notna()
    & ~df["nha_mang"].isin(nha_mang_hop_le | {"Không xác định"})
)

print("Sau xử lý:")
print(df["nha_mang"].value_counts(dropna=False))


XỬ LÝ NHÀ MẠNG
Các dòng lỗi trước khi xử lý:
 STT          ID nha_mang
7807 Hai Nguyen-        0
7808       nknow        0
Sau xử lý:
nha_mang
Viettel           11710
Vinaphone          3141
Mobifone           2512
Khác               1613
Không xác định        2
Name: count, dtype: int64


### Kết quả: Chuẩn hóa nhà mạng

- Hai giá trị 0 được thay bằng `Không xác định`.
- Không suy đoán nhà mạng thực tế khi dữ liệu nguồn để trống.

In [12]:
print_header("CELL 4: KIỂM TRA TÊN KHÓA HỌC LẠ")

print("Tên khóa học lạ và mã khóa tương ứng:")
for course in df.loc[df["khoa_hoc_la"], "quan_tam_khoa_hoc"].unique():
    sub = df[df["quan_tam_khoa_hoc"] == course]
    ma = sub["quan_tam_khoa_hoc_1"].dropna().unique()
    print(f"{course} ({len(sub)} dòng): {list(ma)}")


KIỂM TRA TÊN KHÓA HỌC LẠ
Tên khóa học lạ và mã khóa tương ứng:
Thu hồi (1 dòng): []
AI (22 dòng): ['Khóa G']
IT Fresher PNK (8 dòng): []
Business Intelligence (4 dòng): []


In [13]:
print_header("CELL 4: BỔ SUNG MAPPING KHÓA G")

mapping_chuan["AI"] = "Khóa G"
df["khoa_hoc_du_kien"] = df["quan_tam_khoa_hoc"].map(mapping_chuan)
df["khoa_hoc_la"] = (
    df["quan_tam_khoa_hoc"].notna()
    & ~df["quan_tam_khoa_hoc"].isin(mapping_chuan)
)
df["loi_mapping_khoa"] = (
    df["khoa_hoc_du_kien"].notna()
    & df["quan_tam_khoa_hoc_1"].notna()
    & (df["khoa_hoc_du_kien"] != df["quan_tam_khoa_hoc_1"])
)


BỔ SUNG MAPPING KHÓA G


In [14]:
print_header("CELL 4: GHI CHÚ KHÓA CHƯA XÁC ĐỊNH")

df["quan_tam_khoa_hoc_ghichu"] = ""
df.loc[df["quan_tam_khoa_hoc"] == "Thu hồi", "quan_tam_khoa_hoc_ghichu"] = (
    "Có thể là trạng thái lead, cần nghiệp vụ xác nhận"
)
df.loc[
    df["khoa_hoc_la"] & (df["quan_tam_khoa_hoc"] != "Thu hồi"),
    "quan_tam_khoa_hoc_ghichu",
] = "Khóa học mới, cần nghiệp vụ xác nhận"

print(f"Còn {df['khoa_hoc_la'].sum()} dòng chưa xác định mã khóa.")
print(df.loc[df["khoa_hoc_la"], "quan_tam_khoa_hoc"].value_counts())


GHI CHÚ KHÓA CHƯA XÁC ĐỊNH
Còn 13 dòng chưa xác định mã khóa.
quan_tam_khoa_hoc
IT Fresher PNK           8
Business Intelligence    4
Thu hồi                  1
Name: count, dtype: int64


### Kết quả: Kiểm tra tên khóa học

- `AI` xuất hiện 22 lần và đều có mã nguồn `Khóa G`; giữ lại cặp dữ liệu này.
- `TechX Cetificate` là lỗi chính tả, được chuẩn hóa thành `TechX Certificate`.
- `IT Fresher PNK`, `Business Intelligence` và `Thu hồi` còn 13 dòng, chưa có mã đối chiếu nên cần nghiệp vụ xác nhận.

In [15]:
print_header("CELL 5: KIỂM TRA MÃ KHÓA BỊ GÁN LỆCH")

print("Các dòng lỗi trước khi xử lý:")
print(df.loc[df["loi_mapping_khoa"], [
    "STT",
    "ID",
    "quan_tam_khoa_hoc",
    "quan_tam_khoa_hoc_1",
    "khoa_hoc_du_kien",
]].to_string(index=False))


KIỂM TRA MÃ KHÓA BỊ GÁN LỆCH
Các dòng lỗi trước khi xử lý:
  STT                           ID quan_tam_khoa_hoc quan_tam_khoa_hoc_1 khoa_hoc_du_kien
17230    Phạm Huy-Trương Thị Huyền         Part time              Khóa C           Khóa B
17317 Thảo Nguyên-Trương Thị Huyền           Fukuoka              Khóa F           Khóa C
18829   Lê Anh Quân-Đinh Thị Nhung      Data Analyst              Khóa D           Khóa F


In [16]:
print_header("CELL 5: CẬP NHẬT MÃ KHÓA")

mask = df["loi_mapping_khoa"]
df.loc[mask, "quan_tam_khoa_hoc_1_goc"] = df.loc[
    mask, "quan_tam_khoa_hoc_1"
]
df.loc[mask, "quan_tam_khoa_hoc_1"] = df.loc[
    mask, "khoa_hoc_du_kien"
]
df["loi_mapping_khoa"] = (
    df["khoa_hoc_du_kien"].notna()
    & df["quan_tam_khoa_hoc_1"].notna()
    & (df["khoa_hoc_du_kien"] != df["quan_tam_khoa_hoc_1"])
)

print(f"Lỗi mapping còn lại: {df['loi_mapping_khoa'].sum()} dòng")


CẬP NHẬT MÃ KHÓA
Lỗi mapping còn lại: 0 dòng


### Kết quả: Sửa mã khóa bị gán lệch

- Có 3 dòng mà `quan_tam_khoa_hoc_1` không khớp với mã suy ra từ câu trả lời gốc.
- Mã cũ được lưu ở `quan_tam_khoa_hoc_1_goc` trước khi cập nhật để tiện đối chiếu.

In [17]:
print_header("CELL 6: KIỂM TRA NGÀY ÂM")

neg = df[df["loi_ngay_am"]]
big = neg[neg["so_ngay_tu_van_den_chot"] <= -20]
small = neg[neg["so_ngay_tu_van_den_chot"] > -20]

print(f"Tổng số dòng âm: {len(neg)}")
print(f"Lệch từ 20 ngày trở lên: {len(big)}")
print(f"Lệch dưới 20 ngày: {len(small)}")
print("Ngày tư vấn phổ biến trong nhóm lệch lớn:")
print(big["thoi_gian_chuyen_sang_tu_van"].dt.date.value_counts().head(5))


KIỂM TRA NGÀY ÂM
Tổng số dòng âm: 164
Lệch từ 20 ngày trở lên: 53
Lệch dưới 20 ngày: 111
Ngày tư vấn phổ biến trong nhóm lệch lớn:
thoi_gian_chuyen_sang_tu_van
2024-07-10    9
2024-06-18    5
2024-10-28    5
2024-08-08    4
2024-07-02    3
Name: count, dtype: int64


In [18]:
print_header("CELL 6: XỬ LÝ NGÀY ÂM")

df["so_ngay_tu_van_den_chot_goc"] = df["so_ngay_tu_van_den_chot"]
df["ghi_chu_ngay_am"] = ""
df.loc[df["loi_ngay_am"], "ghi_chu_ngay_am"] = (
    "Ngày tư vấn sau ngày chốt; loại khỏi phép tính thời gian"
)
df.loc[df["loi_ngay_am"], "so_ngay_tu_van_den_chot"] = np.nan

print("Số ngày âm còn lại:", (df["so_ngay_tu_van_den_chot"] < 0).sum())
print(df["so_ngay_tu_van_den_chot"].describe())


XỬ LÝ NGÀY ÂM
Số ngày âm còn lại: 0
count    891.000000
mean      29.742985
std       33.539716
min        0.000000
25%        7.000000
50%       20.000000
75%       37.000000
max      168.000000
Name: so_ngay_tu_van_den_chot, dtype: float64


### Kết quả: Kiểm tra và xử lý ngày âm

- Có 164 dòng có `so_ngay_tu_van_den_chot` âm. Một số lượng lớn hồ sơ có cùng ngày chuyển tư vấn, gợi ý khả năng hệ thống cập nhật hàng loạt.
- Không có căn cứ để hoán đổi ngày hoặc lấy trị tuyệt đối. Giữ cột ngày gốc, lưu giá trị ban đầu ở cột `_goc` và loại khoảng cách ngày âm khỏi phép tính.

In [19]:
print_header("CELL 7: XỬ LÝ TÊN RỖNG")

print(
    "Tư vấn viên rỗng:",
    (df["tu_van_vien"].fillna("").str.strip() == "").sum(),
)
print(
    "Khách hàng rỗng:",
    (df["khach_hang"].fillna("").str.strip() == "").sum(),
)

df["tu_van_vien"] = df["tu_van_vien"].replace("", np.nan).fillna("Chưa xác định")
df["khach_hang"] = df["khach_hang"].replace("", np.nan).fillna("Chưa xác định")
df["loi_ten_rong"] = (
    (df["khach_hang"] == "Chưa xác định")
    | (df["tu_van_vien"] == "Chưa xác định")
)

print("Số dòng chưa xác định:", df["loi_ten_rong"].sum())
print(df["tu_van_vien"].value_counts().head(3))


XỬ LÝ TÊN RỖNG
Tư vấn viên rỗng: 704
Khách hàng rỗng: 12
Số dòng chưa xác định: 709
tu_van_vien
Trương Thị Huyền    1455
Bùi Ngọc Anh        1075
Cao Thảo My         1041
Name: count, dtype: int64


### Kết quả: Gán nhãn tên còn thiếu

- ID nguồn thiếu tên tư vấn viên hoặc khách hàng; đây không phải lỗi tách chuỗi.
- Gán `Chưa xác định` để có thể lọc và phân tích, đồng thời giữ cờ `loi_ten_rong` để truy vết.

In [20]:
so_loi_ban_dau = df_summary.set_index("Cờ lỗi")["Số dòng lỗi"]
so_ai = (df["quan_tam_khoa_hoc"] == "AI").sum()
so_khoa_chua_ro = df["khoa_hoc_la"].sum()

ket_qua = pd.DataFrame([
    ["nha_mang", "Giá trị 0", so_loi_ban_dau["loi_nha_mang"], "Thay bằng Không xác định", "Dễ"],
    ["quan_tam_khoa_hoc", "AI, mã gốc Khóa G", so_ai, "Giữ mã gốc, xác nhận ý nghĩa", "Xác nhận"],
    ["quan_tam_khoa_hoc", "Tên khóa chưa có mã", so_khoa_chua_ro, "Chờ nghiệp vụ xác nhận", "Khó"],
    ["quan_tam_khoa_hoc_1", "Mã khóa gán lệch", so_loi_ban_dau["loi_mapping_khoa"], "Sửa theo câu trả lời gốc", "Dễ"],
    ["so_ngay_tu_van_den_chot", "Ngày âm", so_loi_ban_dau["loi_ngay_am"], "Không dùng để tính thời gian", "Trung bình"],
    ["khach_hang / tu_van_vien", "ID thiếu tên", df["loi_ten_rong"].sum(), "Gán nhãn Chưa xác định", "Dễ"],
], columns=["Cột dữ liệu", "Nguyên nhân", "Số dòng", "Cách xử lý", "Ghi chú"])

ket_qua

,Cột dữ liệu,Nguyên nhân,Số dòng,Cách xử lý,Ghi chú
0,nha_mang,Giá trị 0,2,Thay bằng Không xác định,Dễ
1,quan_tam_khoa_hoc,"AI, mã gốc Khóa G",22,"Giữ mã gốc, xác nhận ý nghĩa",Xác nhận
2,quan_tam_khoa_hoc,Tên khóa chưa có mã,13,Chờ nghiệp vụ xác nhận,Khó
3,quan_tam_khoa_hoc_1,Mã khóa gán lệch,3,Sửa theo câu trả lời gốc,Dễ
4,so_ngay_tu_van_den_chot,Ngày âm,164,Không dùng để tính thời gian,Trung bình
5,khach_hang / tu_van_vien,ID thiếu tên,709,Gán nhãn Chưa xác định,Dễ


## Khóa G và khóa chưa xác định

### Khóa G là gì?

`Khóa G` là mã có sẵn trong file nguồn. Cả 22 dòng có câu trả lời `AI` đều đi cùng mã này, nên notebook giữ nguyên cặp dữ liệu khi chuẩn hóa.

Dữ liệu chưa cho biết tên đầy đủ hoặc ý nghĩa nghiệp vụ của `Khóa G`. Cần hỏi bộ phận phụ trách trước khi đổi mã thành tên sản phẩm.

### 13 dòng chưa có mã đối chiếu

| Giá trị gốc | Số dòng |
|---|---:|
| IT Fresher PNK | 8 |
| Business Intelligence | 4 |
| Thu hồi | 1 |

Các dòng này được giữ lại và gắn ghi chú để nghiệp vụ xác nhận, không tự đoán mã.

## Tên khách hàng và tư vấn viên còn thiếu

- 704 dòng chưa xác định tư vấn viên.
- 12 dòng chưa xác định khách hàng.
- 7 dòng thiếu cả hai tên; tổng số dòng thiếu ít nhất một tên là **709**, không phải 716.

Các tên thiếu được gán nhãn `Chưa xác định`, không tự suy ra tên từ ID.

In [ ]:
print_header("XUẤT FILE ĐÃ XỬ LÝ")

df.to_excel(OUTPUT_XLSX, index=False)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Đã lưu file đã xử lý: {OUTPUT_XLSX}")
print(f"Đã lưu file CSV: {OUTPUT_CSV}")


XUẤT FILE ĐÃ XỬ LÝ
Đã lưu file đã xử lý: Courses_of_Interest_Processed.xlsx
Đã lưu file CSV: Courses_of_Interest_Processed.csv


: 

### Kết quả: Xuất dữ liệu đã xử lý

File Excel và CSV chứa dữ liệu gốc, các cột đã chuẩn hóa và các cột `_goc`/ghi chú để tra cứu thay đổi.

## 🏁 TỔNG KẾT

| Cột | Lỗi | Số dòng | Dễ/khó sửa |
|---|---|---|---|
| `nha_mang` | Giá trị 0 | 2 | Dễ |
| `quan_tam_khoa_hoc` | 'AI' chưa có mapping | 22 | Dễ (đã có mã đi kèm 100%) |
| `quan_tam_khoa_hoc` | 'IT Fresher PNK' / 'Business Intelligence' / 'Thu hồi' | 13 | Khó — cần nghiệp vụ xác nhận |
| `quan_tam_khoa_hoc_1` | Mã khóa gán lệch | 3 | Dễ — có nguồn đối chiếu |
| `so_ngay_tu_van_den_chot` | Ngày âm (nghi hệ thống cập nhật hàng loạt) | 164 | Trung bình — không tự suy diễn được thứ tự thật |
| `khach_hang` / `tu_van_vien` | Tên rỗng do ID thiếu tư vấn viên | 709 | Dễ — do thiếu dữ liệu gốc |

**13 dòng khóa học lạ còn lại** (`IT Fresher PNK`, `Business Intelligence`, `Thu hồi`) là nhóm duy nhất chưa thể tự động xử lý dứt điểm vì không có mã khóa đối chiếu — đã gắn ghi chú trong cột `quan_tam_khoa_hoc_ghichu` để đội nghiệp vụ xác nhận thêm.